## FULL PROCESSING PIPELINE 

In [91]:
import pandas as pd

df = pd.read_csv('sample.csv', index_col=0)

print(df.shape)         
print(df.columns)       
print(df['keywords'].value_counts())

(406, 2)
Index(['content', 'keywords'], dtype='str')
keywords
Prejudiciële bevoegheid (Grondwettelijk Hof)     258
Vernietigingsbevoegdheid (Grondwettelijk Hof)    148
Name: count, dtype: int64


In [93]:
import re

In [94]:
def clean_text(text):
    text = str(text)
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = re.sub(r"\\[a-zA-Z]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

 

    return text


df['content_clean'] = df['content'].apply(clean_text)


In [95]:
import ftfy

df['content_clean'] = df['content_clean'].apply(ftfy.fix_text)

sample = df['content_clean'].iloc[0][:200]
print(sample)

"La Cour constitutionnelle,composC)e des prC)sidents F. DaoC;t et A. Alen, et des juges L. Lavrysen, J.-P. Snappe, J.-P. Moerman, E. Derycke, T. Merckx-Van Goey, P. Nihoul, T. Giet, R. Leysen, J. Moer


In [96]:
from lingua import Language, LanguageDetectorBuilder

detector = LanguageDetectorBuilder.from_languages(
    Language.FRENCH, Language.DUTCH, Language.GERMAN
).build()

def detect_lang(text):
    result = detector.detect_language_of(text[:500])
    return result.name if result else "UNKNOWN"

df['language'] = df['content_clean'].apply(detect_lang)
print(df['language'].value_counts())

language
DUTCH     150
FRENCH    134
GERMAN    122
Name: count, dtype: int64


In [ ]:
def fix_encoding_artifacts(text):
    result = []
    i = 0
    text = str(text)
    while i < len(text):
        if text[i] == 'C' and i + 1 < len(text):
            next_char = text[i + 1]
            code = ord(next_char) + 0xC0
            if 0xC0 <= code <= 0xFF:
                result.append(chr(code))
                i += 2
                continue
        result.append(text[i])
        i += 1
    return "".join(result)

df['content_clean'] = df['content_clean'].apply(fix_encoding_artifacts)

# verify
print(df['content_clean'].iloc[0][:300])

" F. Daoût et A. Alen, et des juges L. Lavrysen, J.-P. Snappe, J.-P. Moerman, E. Derycke, T. Merckx-Van Goey, P. Nihoul, T. Giet, R. Leysen, J. Moerman et M. PC\"ques, assistée du greffier P.-Y. Dutilleux, présidée par le président F. Daoût,après en avoir délibéré, rend l'arrêt suivant :I. Objet du 


In [65]:
import re

def final_clean(text):
    text = str(text)
    text = text.strip('"').strip("'")
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['content_clean'] = df['content_clean'].apply(final_clean)
print(df['content_clean'].iloc[0][:300])

F. Daoût et A. Alen, et des juges L. Lavrysen, J.-P. Snappe, J.-P. Moerman, E. Derycke, T. Merckx-Van Goey, P. Nihoul, T. Giet, R. Leysen, J. Moerman et M. PC\"ques, assistée du greffier P.-Y. Dutilleux, présidée par le président F. Daoût,après en avoir délibéré, rend l'arrêt suivant :I. Objet du re


In [66]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("Max sequence length:", model.max_seq_length)  
print("Embedding dim:", model.get_sentence_embedding_dimension()) 

c:\Users\HP\Desktop\Bacancy\AI_ML\Deep_Learning\dl_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\HP\Desktop\Bacancy\AI_ML\Deep_Learning\dl_env\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Devel

Max sequence length: 128
Embedding dim: 384


In [67]:
def chunk_text(text, chunk_size=100, overlap=20):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start += chunk_size - overlap 
    return chunks

# preview
sample_chunks = chunk_text(df['content_clean'].iloc[0])
print(f"Document word count : {len(df['content_clean'].iloc[0].split())}")
print(f"Number of chunks    : {len(sample_chunks)}")
print(f"First chunk         : {sample_chunks[0][:150]}")

Document word count : 4688
Number of chunks    : 59
First chunk         : F. Daoût et A. Alen, et des juges L. Lavrysen, J.-P. Snappe, J.-P. Moerman, E. Derycke, T. Merckx-Van Goey, P. Nihoul, T. Giet, R. Leysen, J. Moerman 


In [68]:
import numpy as np
from tqdm import tqdm

def embed_document(text, model, chunk_size=100, overlap=20):
    chunks = chunk_text(text, chunk_size, overlap)
    chunk_embeddings = model.encode(chunks, batch_size=32, show_progress_bar=False)
    return np.mean(chunk_embeddings, axis=0)

embeddings = []

for text in tqdm(df['content_clean'], desc="Embedding documents"):
    emb = embed_document(text, model)
    embeddings.append(emb)

embeddings = np.array(embeddings)
print("Embeddings shape:", embeddings.shape)  # (406, 384)

Embedding documents: 100%|██████████| 406/406 [22:04<00:00,  3.26s/it]

Embeddings shape: (406, 384)


## MODEL TRAINING 

In [88]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np

le = LabelEncoder()
y = le.fit_transform(df['keywords'])   # 0 or 1

print("Classes:", le.classes_)
print("Label distribution:", np.bincount(y))

Classes: ['Prejudiciële bevoegheid (Grondwettelijk Hof)'
 'Vernietigingsbevoegdheid (Grondwettelijk Hof)']
Label distribution: [258 148]


In [89]:
y.shape

(406,)

In [108]:
import tensorflow as tf

X_emb = embeddings  

X_emb_train, X_emb_test, y_emb_train, y_emb_test = train_test_split(
    X_emb, y, test_size=0.2, random_state=42, stratify=y
)

model_emb = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(384,)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model_emb.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_emb.summary()

history_emb = model_emb.fit(
    X_emb_train, y_emb_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

loss, acc = model_emb.evaluate(X_emb_test, y_emb_test)
print(f"\nModel 1 (Embeddings) → Test Accuracy: {acc:.4f}")

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 256)            │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 131,585 (514.00 KB)

 Trainable params: 131,585 (514.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.6117 - loss: 0.6390 - val_accuracy: 0.6970 - val_loss: 0.5674
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6667 - loss: 0.5761 - val_accuracy: 0.6970 - val_loss: 0.5195
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7320 - loss: 0.5134 - val_accuracy: 0.7273 - val_loss: 0.4760
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7973 - loss: 0.4668 - val_accuracy: 0.6970 - val_loss: 0.4911
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7973 - loss: 0.4188 - val_accuracy: 0.7273 - val_loss: 0.4744
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8041 - loss: 0.3831 - val_accuracy: 0.8182 - val_loss: 0.3743
Epoch 7/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8729 - loss: 0.3391 - val_accuracy: 0.8182 - val_loss: 0.3675
Epoch 8/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9038 - loss: 0.2797 - val_accuracy: 0.8182 - v

## WITHOUT EMBEDDING

In [103]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

X_text = df['content_clean'].astype(str).values

X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    sublinear_tf=True      
)

X_train_tfidf = tfidf.fit_transform(X_text_train)
X_test_tfidf  = tfidf.transform(X_text_test)

print("TF-IDF matrix shape (train):", X_train_tfidf.shape)
print("TF-IDF matrix shape (test) :", X_test_tfidf.shape)

TF-IDF matrix shape (train): (324, 20000)
TF-IDF matrix shape (test) : (82, 20000)


In [104]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_text_train)

y_pred_lr = lr.predict(X_test_tfidf)
acc_lr = accuracy_score(y_text_test, y_pred_lr)

print(f"Logistic Regression → Test Accuracy: {acc_lr:.4f}")
print(classification_report(y_text_test, y_pred_lr, target_names=le.classes_))

Logistic Regression → Test Accuracy: 0.9268
                                               precision    recall  f1-score   support

 Prejudiciële bevoegheid (Grondwettelijk Hof)       0.90      1.00      0.95        52
Vernietigingsbevoegdheid (Grondwettelijk Hof)       1.00      0.80      0.89        30

                                     accuracy                           0.93        82
                                    macro avg       0.95      0.90      0.92        82
                                 weighted avg       0.93      0.93      0.92        82



In [ ]:
svm = SVC(kernel='linear', C=1.0, random_state=42)
svm.fit(X_train_tfidf, y_text_train)

y_pred_svm = svm.predict(X_test_tfidf)
acc_svm = accuracy_score(y_text_test, y_pred_svm)

print(f"SVM (linear kernel) → Test Accuracy: {acc_svm:.4f}")
print(classification_report(y_text_test, y_pred_svm, target_names=le.classes_))

SVM (linear kernel) → Test Accuracy: 0.9390
                                               precision    recall  f1-score   support

 Prejudiciële bevoegheid (Grondwettelijk Hof)       0.94      0.96      0.95        52
Vernietigingsbevoegdheid (Grondwettelijk Hof)       0.93      0.90      0.92        30

                                     accuracy                           0.94        82
                                    macro avg       0.94      0.93      0.93        82
                                 weighted avg       0.94      0.94      0.94        82



In [109]:
# ── FINAL COMPARISON ──────────────────────────────────────────────────
print("=" * 45)
print(f"Model 1  — Embedding + DNN       : {acc:.4f}")
print(f"Model 2a — TF-IDF + Logistic Reg : {acc_lr:.4f}")
print(f"Model 2b — TF-IDF + SVM          : {acc_svm:.4f}")
print("=" * 45)

Model 1  — Embedding + DNN       : 0.9146
Model 2a — TF-IDF + Logistic Reg : 0.9268
Model 2b — TF-IDF + SVM          : 0.9390
